# 20. Brand Portfolio Analysis

## Objective

This notebook examines the composition, structure and positioning of selected fragrance brand portfolios. The analysis focuses on manufacturers and their fragrance offerings rather than retailers or marketplace sellers.

The objective is to identify how selected luxury and niche brands differ in portfolio size, gender classification, fragrance accords, product concentration, fragrance families, notes and other product characteristics.

## Data Sources

Three sources are used:

- **Fragrantica:** Main source for fragrance catalogue records, brand names, gender classifications, main accords, community ratings and rating counts.
- **Scento:** Supplementary fragrance database providing concentration, launch year, fragrance families, character, seasons, times of day, notes and perfumer information.
- **10ML Fragrance Index API:** Additional source for individual fragrance profiles, including house, launch year, gender, family, accords, key notes and character.

## Brand Selection

The initial candidate sample contains 14 brands:

- **Luxury:** Dior, Guerlain, Chanel, Hermes, Mugler, Yves Saint Laurent and Tom Ford.
- **Niche:** Nicolai Parfumeur Createur, Nasomatto, Memo Paris, Byredo, 27 87, Maison Francis Kurkdjian and Serge Lutens.

The final sample will be determined after evaluating data availability across the sources.

## Analytical Approach

The analysis will examine:

1. Brand portfolio size and composition.
2. Gender classification and portfolio distribution.
3. Accord frequency, diversity and concentration.
4. Fragrance concentration and product characteristics.
5. Launch-year distribution and portfolio development.
6. Similarities and differences between individual brands.
7. Aggregate patterns across the selected luxury and niche groups.

## Methodological Considerations

The selected brands constitute a purposive sample and are not intended to represent the entire luxury or niche fragrance industry.

The datasets have different catalogue coverage and may contain overlapping fragrance records. Source-level counts will therefore be reported separately. Records will only be combined when fragrance identity can be established reliably.

The Fragrantica source table will remain unchanged. All transformations and analytical calculations will be performed in separate DataFrames.

In [0]:
%pip install perfume-api

In [0]:
# 1. Load data sources.

import requests
import pandas as pd
from io import StringIO
from pyspark.sql import functions as F

# Fragrantica
df_fragrantica = spark.table(
    "main.fragrance_analytics.fragrantica_perfumes"
)

print("Fragrantica perfumes:", df_fragrantica.count())

# Scento
SCENTO_BASE = (
    "https://huggingface.co/datasets/"
    "scento/perfume-database/resolve/main"
)

HEADERS = {
    "User-Agent": "FragranceRecommendationPlatform/0.1"
}

SOURCE_FILES = {
    "perfumes": f"{SCENTO_BASE}/perfumes.csv",
    "houses": f"{SCENTO_BASE}/houses.csv",
    "materials": f"{SCENTO_BASE}/materials.csv"
}

scento_data = {}

for name, url in SOURCE_FILES.items():
    response = requests.get(
        url,
        headers=HEADERS,
        timeout=30
    )
    response.raise_for_status()

    scento_data[name] = pd.read_csv(
        StringIO(response.text)
    )

    print(
        f"{name.upper()}: "
        f"{len(scento_data[name]):,} rows, "
        f"{len(scento_data[name].columns)} columns"
    )

In [0]:
# 2. Define candidate brands for portfolio analysis.

brand_groups = {
    "Luxury": [
        "Dior",
        "Guerlain",
        "Chanel",
        "Hermes",
        "Mugler",
        "Yves Saint Laurent",
        "Tom Ford"
    ],
    "Niche": [
        "Nicolai Parfumeur Createur",
        "Nasomatto",
        "Memo Paris",
        "Byredo",
        "Maison Francis Kurkdjian",
        "Serge Lutens"
    ]
}

brand_candidates = [
    (brand, group)
    for group, brands in brand_groups.items()
    for brand in brands
]

print(f"Candidate brands: {len(brand_candidates)}")

for group, brands in brand_groups.items():
    print(f"\n{group}: {len(brands)}")
    for brand in brands:
        print(f"  - {brand}")

In [0]:
# 3. Check candidate brand coverage in Fragrantica and Scento.

candidate_df = pd.DataFrame(
    brand_candidates,
    columns=["brand_name", "brand_group"]
)

candidate_df["brand_key"] = (
    candidate_df["brand_name"].str.strip().str.casefold()
)

# Fragrantica counts.
fragrantica_counts = (
    df_fragrantica
    .filter(F.col("brand_name").isNotNull())
    .withColumn(
        "brand_key",
        F.lower(F.trim(F.col("brand_name")))
    )
    .groupBy("brand_key")
    .agg(
        F.countDistinct("id").alias("fragrantica_count")
    )
    .toPandas()
)

# Scento counts.
df_scento = scento_data["perfumes"].copy()

df_scento["brand_key"] = (
    df_scento["house"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

scento_counts = (
    df_scento
    .dropna(subset=["brand_key"])
    .groupby("brand_key")["id"]
    .nunique()
    .reset_index(name="scento_count")
)

# Combine coverage results.
brand_coverage = (
    candidate_df
    .merge(fragrantica_counts, on="brand_key", how="left")
    .merge(scento_counts, on="brand_key", how="left")
)

brand_coverage["fragrantica_count"] = (
    brand_coverage["fragrantica_count"].fillna(0).astype(int)
)

brand_coverage["scento_count"] = (
    brand_coverage["scento_count"].fillna(0).astype(int)
)

brand_coverage["both_datasets"] = (
    (brand_coverage["fragrantica_count"] > 0) &
    (brand_coverage["scento_count"] > 0)
)

brand_coverage["combined_source_records"] = (
    brand_coverage["fragrantica_count"] +
    brand_coverage["scento_count"]
)

brand_coverage = brand_coverage.sort_values(
    ["both_datasets", "combined_source_records"],
    ascending=[False, False]
).reset_index(drop=True)

display(
    brand_coverage[
        [
            "brand_name",
            "brand_group",
            "fragrantica_count",
            "scento_count",
            "both_datasets",
            "combined_source_records"
        ]
    ]
)

In [0]:
# 4. Inspect alternative brand and house names.

search_terms = [
    "chanel",
    "herm",
    "nicolai",
    "2787",
    "27 87",
    "serge lutens",
    "nasomatto"
]

print("FRAGRANTICA BRAND NAMES")

fragrantica_brands = (
    df_fragrantica
    .select("brand_name")
    .distinct()
    .filter(F.col("brand_name").isNotNull())
)

for term in search_terms:
    print(f"\nSearch: {term}")
    display(
        fragrantica_brands
        .filter(F.lower(F.col("brand_name")).contains(term))
        .orderBy("brand_name")
    )

print("\nSCENTO HOUSE NAMES")

scento_houses = sorted(
    scento_data["houses"]["name"].dropna().unique()
)

for term in search_terms:
    matches = [
        house for house in scento_houses
        if term in house.casefold()
    ]
    print(f"\nSearch: {term}")
    print(matches)

In [0]:
# 5. Explore candidate brands in the 10ML Fragrance Index.

from perfume_api import search_perfumes

api_search_results = {}
api_coverage_rows = []

for brand, group in brand_candidates:

    try:
        result = search_perfumes(
            brand,
            limit=5
        )

        api_search_results[brand] = result

        perfumes = result.get("results", [])

        api_coverage_rows.append({
            "brand_name": brand,
            "brand_group": group,
            "api_status": "OK",
            "api_search_total": result.get("total", 0),
            "returned_records": len(perfumes),
            "returned_houses": ", ".join(
                sorted({
                    perfume.get("house", "")
                    for perfume in perfumes
                    if perfume.get("house")
                })
            ),
            "returned_fragrances": ", ".join(
                perfume.get("name", "")
                for perfume in perfumes
            )
        })

    except Exception as e:

        api_search_results[brand] = None

        api_coverage_rows.append({
            "brand_name": brand,
            "brand_group": group,
            "api_status": str(e),
            "api_search_total": 0,
            "returned_records": 0,
            "returned_houses": "",
            "returned_fragrances": ""
        })

api_coverage = pd.DataFrame(api_coverage_rows)

display(api_coverage)

In [0]:
# 6. Inspect houses available in the 10ML Fragrance Index.

from perfume_api import list_houses

result = list_houses(limit=50)

print(result.keys())

api_houses = result["houses"]

df_api_houses = pd.DataFrame(api_houses)

display(df_api_houses.sort_values("name"))

In [0]:
# 7. Check candidate brand coverage in the 10ML house index.

api_brand_aliases = {
    "Hermes": "Hermès",
    "Nicolai Parfumeur Createur": "Nicolaï"
}

api_brand_coverage = candidate_df.copy()

api_brand_coverage["api_lookup_name"] = (
    api_brand_coverage["brand_name"]
    .replace(api_brand_aliases)
)

api_brand_coverage = api_brand_coverage.merge(
    df_api_houses[
        ["name", "perfumes_in_index"]
    ],
    left_on="api_lookup_name",
    right_on="name",
    how="left"
)

api_brand_coverage = api_brand_coverage.rename(
    columns={
        "name": "api_house_name",
        "perfumes_in_index": "api_perfume_count"
    }
)

display(
    api_brand_coverage[
        [
            "brand_name",
            "brand_group",
            "api_house_name",
            "api_perfume_count"
        ]
    ]
)

In [0]:
# 7. Check candidate brand coverage in the 10ML house index.

api_house_lookup = (
    df_api_houses
    .assign(house_key=lambda x: x["name"].str.casefold())
    .set_index("house_key")
)

api_brand_coverage = candidate_df.copy()

api_brand_coverage["house_key"] = (
    api_brand_coverage["brand_name"].str.casefold()
)

api_brand_coverage["api_house_name"] = (
    api_brand_coverage["house_key"]
    .map(api_house_lookup["name"])
)

api_brand_coverage["api_perfume_count"] = (
    api_brand_coverage["house_key"]
    .map(api_house_lookup["perfumes_in_index"])
)

display(
    api_brand_coverage[
        [
            "brand_name",
            "brand_group",
            "api_house_name",
            "api_perfume_count"
        ]
    ]
)

In [0]:
# 6. Fragrantica portfolio composition by brand and gender.

candidate_brand_names = [
    brand for brand, group in brand_candidates
]

df_fragrantica_candidates = (
    df_fragrantica
    .filter(
        F.col("brand_name").isin(candidate_brand_names)
    )
)

brand_gender_counts = (
    df_fragrantica_candidates
    .groupBy("brand_name", "gender")
    .agg(
        F.countDistinct("id").alias("fragrance_count")
    )
    .orderBy("brand_name", "gender")
)

display(brand_gender_counts)

In [0]:
%sql
DESCRIBE TABLE main.fragrance_analytics.fragrantica_perfumes;

In [0]:
%sql
WITH benchmarks AS (
    SELECT 'men' AS gender, 292.70 AS mean_attention, 4.000 AS mean_rating
    UNION ALL
    SELECT 'unisex', 141.28, 3.985
    UNION ALL
    SELECT 'women', 271.21, 3.968
),

guerlain AS (
    SELECT
        f.id,
        f.fragrance_name_clean AS fragrance,
        f.gender,
        f.rating_value AS rating,
        f.rating_count AS votes,
        f.main_accords
    FROM main.fragrance_analytics.fragrantica_perfumes f
    WHERE LOWER(f.brand_name) = 'guerlain'
      AND LOWER(f.gender) IN ('men', 'unisex', 'women')
      AND f.rating_value IS NOT NULL
      AND f.rating_count IS NOT NULL
)

SELECT
    g.fragrance,
    INITCAP(g.gender) AS gender,
    g.rating,
    g.votes,
    CASE
        WHEN g.rating >= b.mean_rating
         AND g.votes >= b.mean_attention
            THEN 'Standout Profiles'

        WHEN g.rating >= b.mean_rating
         AND g.votes < b.mean_attention
            THEN 'Hidden Favourites'

        WHEN g.rating < b.mean_rating
         AND g.votes >= b.mean_attention
            THEN 'Widely Explored'

        ELSE 'Quiet Profiles'
    END AS performance_segment,
    g.main_accords
FROM guerlain g
JOIN benchmarks b
    ON LOWER(g.gender) = b.gender
ORDER BY
    CASE LOWER(g.gender)
        WHEN 'men' THEN 1
        WHEN 'unisex' THEN 2
        WHEN 'women' THEN 3
    END,
    performance_segment,
    g.votes DESC;

In [0]:
# ACCORD DATA PREPARATION
# Parse the source's main_accords text into an ordered array. The order is
# meaningful: the first item is the dominant accord, the first two items
# form the dyad, and the first three form the triad.
# Apply identical parsing and normalization to Nasomatto and the industry
# reference population so their profiles can be compared consistently.


# ============================================================
# GUERLAIN — FRAGRANCE-LEVEL TRIAD SEGMENTATION TABLE
# Uses the established Notebook 17 triad methodology.
# ============================================================

import numpy as np
from pyspark.sql import functions as F

# ------------------------------------------------------------
# 1. PREPARE ORDERED TRIADS FOR THE FULL FRAGRANTICA DATASET
# ------------------------------------------------------------

accords_clean = F.trim(F.col("main_accords"))

valid_accords = (
    F.col("main_accords").isNotNull()
    & (accords_clean != "")
    & (~F.lower(accords_clean).isin("null", "none", "nan", "n/a", "[]"))
)

accord_array = F.from_json(
    F.regexp_replace(
        F.regexp_replace(F.col("main_accords"), "'", '"'),
        r"\bNone\b",
        "null"
    ),
    "array<string>"
)

triad_source = (
    df_fragrantica
    .filter(
        valid_accords
        & F.col("rating_value").isNotNull()
        & F.col("rating_count").isNotNull()
        & (F.col("rating_count") >= 0)
    )
    .withColumn("accords", accord_array)
    .withColumn(
        "accords",
        F.transform("accords", lambda x: F.lower(F.trim(x)))
    )
)

for position, column_name in [
    (0, "dominant_accord"),
    (1, "second_accord"),
    (2, "third_accord")
]:
    triad_source = triad_source.withColumn(
        column_name,
        F.when(
            (F.size("accords") > position)
            & F.col("accords")[position].isNotNull()
            & (F.trim(F.col("accords")[position]) != ""),
            F.col("accords")[position]
        )
    )

triad_source = triad_source.filter(
    F.col("dominant_accord").isNotNull()
    & F.col("second_accord").isNotNull()
    & F.col("third_accord").isNotNull()
)

# ------------------------------------------------------------
# 2. AGGREGATE TRIADS AND APPLY THE EXISTING SUPPORT RULE
# ------------------------------------------------------------

triad_stats = (
    triad_source
    .groupBy(
        "dominant_accord",
        "second_accord",
        "third_accord"
    )
    .agg(
        F.countDistinct("id").alias("fragrance_count"),
        F.avg("rating_count").alias("mean_votes"),
        F.avg("rating_value").alias("mean_rating")
    )
    .filter(F.col("fragrance_count") >= 50)
)

triads_pdf = triad_stats.toPandas()

# ------------------------------------------------------------
# 3. RECREATE THE NOTEBOOK 17 TRIAD SEGMENTS
# ------------------------------------------------------------

median_attention = triads_pdf["mean_votes"].median()
median_rating = triads_pdf["mean_rating"].median()

triads_pdf["segment"] = np.select(
    [
        (triads_pdf["mean_votes"] >= median_attention)
        & (triads_pdf["mean_rating"] >= median_rating),

        (triads_pdf["mean_votes"] < median_attention)
        & (triads_pdf["mean_rating"] >= median_rating),

        (triads_pdf["mean_votes"] >= median_attention)
        & (triads_pdf["mean_rating"] < median_rating)
    ],
    [
        "Standout Profiles",
        "Hidden Favourites",
        "Widely Explored"
    ],
    default="Quiet Profiles"
)

print(f"Eligible triads: {len(triads_pdf):,}")
print(f"Median mean votes: {median_attention:.3f}")
print(f"Median mean rating: {median_rating:.4f}")

# Create the profile-to-segment lookup.
triad_lookup = spark.createDataFrame(
    triads_pdf[
        [
            "dominant_accord",
            "second_accord",
            "third_accord",
            "segment"
        ]
    ]
)

# ------------------------------------------------------------
# 4. MATCH GUERLAIN FRAGRANCES TO THEIR TRIAD SEGMENTS
# ------------------------------------------------------------

guerlain_triads = (
    triad_source
    .filter(
        F.lower(F.trim(F.col("brand_name"))) == "guerlain"
    )
    .join(
        triad_lookup,
        on=[
            "dominant_accord",
            "second_accord",
            "third_accord"
        ],
        how="inner"
    )
    .select(
        F.initcap("gender").alias("Gender"),

        F.concat_ws(
            " + ",
            F.initcap("dominant_accord"),
            F.initcap("second_accord"),
            F.initcap("third_accord")
        ).alias("Triad"),

        F.col("fragrance_name_clean").alias("Fragrance"),
        F.col("rating_value").alias("Rating"),
        F.col("rating_count").alias("Votes"),
        F.col("segment").alias("Segment")
    )
    .orderBy(
        "Gender",
        "Triad",
        F.desc("Votes"),
        "Fragrance"
    )
)

display(guerlain_triads)


In [0]:

# ============================================================
# GUERLAIN — FRAGRANCE-LEVEL DYAD SEGMENTATION TABLE
# Uses the established Notebook 17 dyad methodology.
# ============================================================

# ------------------------------------------------------------
# 1. PREPARE ORDERED DYADS FOR THE FULL FRAGRANTICA DATASET
# ------------------------------------------------------------

accords_clean_dyad = F.trim(F.col("main_accords"))

valid_accords_dyad = (
    F.col("main_accords").isNotNull()
    & (accords_clean_dyad != "")
    & (~F.lower(accords_clean_dyad).isin("null", "none", "nan", "n/a", "[]"))
)

accord_array_dyad = F.from_json(
    F.regexp_replace(
        F.regexp_replace(F.col("main_accords"), "'", '"'),
        r"\bNone\b",
        "null"
    ),
    "array<string>"
)

dyad_source = (
    df_fragrantica
    .filter(
        valid_accords_dyad
        & F.col("rating_value").isNotNull()
        & F.col("rating_count").isNotNull()
        & (F.col("rating_count") >= 0)
    )
    .withColumn("accords", accord_array_dyad)
    .withColumn(
        "accords",
        F.transform("accords", lambda x: F.lower(F.trim(x)))
    )
)

for position, column_name in [
    (0, "dominant_accord"),
    (1, "second_accord")
]:
    dyad_source = dyad_source.withColumn(
        column_name,
        F.when(
            (F.size("accords") > position)
            & F.col("accords")[position].isNotNull()
            & (F.trim(F.col("accords")[position]) != ""),
            F.col("accords")[position]
        )
    )

dyad_source = dyad_source.filter(
    F.col("id").isNotNull()
    & F.col("dominant_accord").isNotNull()
    & F.col("second_accord").isNotNull()
)

# ------------------------------------------------------------
# 2. AGGREGATE DYADS AND APPLY THE SEGMENTATION SUPPORT RULE
# ------------------------------------------------------------

dyad_stats = (
    dyad_source
    .groupBy("dominant_accord", "second_accord")
    .agg(
        F.countDistinct("id").alias("fragrance_count"),
        F.avg("rating_count").alias("mean_votes"),
        F.avg("rating_value").alias("mean_rating")
    )
    .filter(F.col("fragrance_count") >= 100)
)

dyads_pdf = dyad_stats.toPandas()

# ------------------------------------------------------------
# 3. RECREATE THE NOTEBOOK 17 DYAD SEGMENTS
# ------------------------------------------------------------

median_attention_dyad = dyads_pdf["mean_votes"].median()
median_rating_dyad = dyads_pdf["mean_rating"].median()

dyads_pdf["segment"] = np.select(
    [
        (dyads_pdf["mean_votes"] >= median_attention_dyad)
        & (dyads_pdf["mean_rating"] >= median_rating_dyad),

        (dyads_pdf["mean_votes"] < median_attention_dyad)
        & (dyads_pdf["mean_rating"] >= median_rating_dyad),

        (dyads_pdf["mean_votes"] >= median_attention_dyad)
        & (dyads_pdf["mean_rating"] < median_rating_dyad)
    ],
    [
        "Standout Profiles",
        "Hidden Favourites",
        "Widely Explored"
    ],
    default="Quiet Profiles"
)

print(f"Eligible dyads: {len(dyads_pdf):,}")
print(f"Median mean votes: {median_attention_dyad:.3f}")
print(f"Median mean rating: {median_rating_dyad:.4f}")

dyad_lookup = spark.createDataFrame(
    dyads_pdf[
        [
            "dominant_accord",
            "second_accord",
            "segment"
        ]
    ]
)

# ------------------------------------------------------------
# 4. MATCH GUERLAIN FRAGRANCES TO THEIR DYAD SEGMENTS
# ------------------------------------------------------------

guerlain_dyads = (
    dyad_source
    .filter(
        F.lower(F.trim(F.col("brand_name"))) == "guerlain"
    )
    .join(
        dyad_lookup,
        on=["dominant_accord", "second_accord"],
        how="inner"
    )
    .select(
        F.initcap("gender").alias("Gender"),

        F.concat_ws(
            " + ",
            F.initcap("dominant_accord"),
            F.initcap("second_accord")
        ).alias("Dyad"),

        F.col("fragrance_name_clean").alias("Fragrance"),
        F.col("rating_value").alias("Rating"),
        F.col("rating_count").alias("Votes"),
        F.col("segment").alias("Segment")
    )
    .orderBy(
        "Gender",
        "Dyad",
        F.desc("Votes"),
        "Fragrance"
    )
)

display(guerlain_dyads)


In [0]:

from pyspark.sql import functions as F

# Prepare the full industry dataset.
accords_clean = F.trim(F.col("main_accords"))

valid_accords = (
    F.col("main_accords").isNotNull()
    & (accords_clean != "")
    & (~F.lower(accords_clean).isin("null", "none", "nan", "n/a", "[]"))
)

accord_array = F.from_json(
    F.regexp_replace(
        F.regexp_replace(F.col("main_accords"), "'", '"'),
        r"\bNone\b",
        "null"
    ),
    "array<string>"
)

industry_accords = (
    df_fragrantica
    .filter(
        valid_accords
        & F.col("rating_value").isNotNull()
        & F.col("rating_count").isNotNull()
        & (F.col("rating_count") >= 0)
        & F.lower(F.trim(F.col("gender"))).isin(
            "men", "unisex", "women"
        )
    )
    .withColumn("accords", accord_array)
    .withColumn(
        "accords",
        F.transform("accords", lambda x: F.lower(F.trim(x)))
    )
    .withColumn("gender_key", F.lower(F.trim(F.col("gender"))))
    .withColumn("dominant_accord", F.get(F.col("accords"), 0))
    .withColumn("second_accord", F.get(F.col("accords"), 1))
    .withColumn("third_accord", F.get(F.col("accords"), 2))
)

# Common reference metrics.
def profile_benchmark(data, profile_columns):
    return (
        data
        .filter(
            F.col("id").isNotNull()
            & F.col("rating_value").isNotNull()
            & F.col("rating_count").isNotNull()
        )
        .groupBy("gender_key", *profile_columns)
        .agg(
            F.countDistinct("id").alias("reference_count"),
            F.avg("rating_value").alias("mean_rating"),
            F.avg("rating_count").alias("mean_votes")
        )
    )

# 1. Dominant accord benchmarks.
dominant_benchmarks = profile_benchmark(
    industry_accords.filter(
        F.col("dominant_accord").isNotNull()
    ),
    ["dominant_accord"]
)

# 2. Ordered dyad benchmarks.
dyad_benchmarks = profile_benchmark(
    industry_accords.filter(
        F.col("dominant_accord").isNotNull()
        & F.col("second_accord").isNotNull()
    ),
    ["dominant_accord", "second_accord"]
)

# 3. Ordered triad benchmarks.
triad_benchmarks = profile_benchmark(
    industry_accords.filter(
        F.col("dominant_accord").isNotNull()
        & F.col("second_accord").isNotNull()
        & F.col("third_accord").isNotNull()
    ),
    ["dominant_accord", "second_accord", "third_accord"]
)

print("Dominant accord benchmarks:", dominant_benchmarks.count())
print("Dyad benchmarks:", dyad_benchmarks.count())
print("Triad benchmarks:", triad_benchmarks.count())


In [0]:
# NASOMATTO PILOT — DATA PREPARATION
# Isolate Nasomatto records from the unchanged Fragrantica source table.
# Keep the fields needed for fragrance identity, gender, accords, ratings
# and vote counts. Check that each fragrance ID occurs only once, then
# retain records with usable rating data and normalize gender for joins.

from pyspark.sql import functions as F

# Nasomatto portfolio
nasomatto_portfolio = (
    df_fragrantica
    .filter(
        F.lower(F.trim(F.col("brand_name"))) == "nasomatto"
    )
    .select(
        "id",
        "fragrance_name_clean",
        "brand_name",
        "gender",
        "rating_value",
        "rating_count",
        "main_accords"
    )
)

# Check that each fragrance appears only once.
assert (
    nasomatto_portfolio.count()
    == nasomatto_portfolio.select("id").distinct().count()
)

# Rated fragrances for portfolio performance analysis.
nasomatto_rated = (
    nasomatto_portfolio
    .filter(
        F.col("rating_value").isNotNull()
        & F.col("rating_count").isNotNull()
        & (F.col("rating_count") >= 0)
    )
    .withColumn(
        "gender_key",
        F.lower(F.trim(F.col("gender")))
    )
)

print("Nasomatto fragrances:", nasomatto_portfolio.count())
print("Rated fragrances:", nasomatto_rated.count())

# Inspect the actual gender distribution.
display(
    nasomatto_portfolio
    .groupBy("gender")
    .agg(
        F.countDistinct("id").alias("fragrance_count")
    )
    .orderBy("gender")
)

In [0]:
accords_clean = F.trim(F.col("main_accords"))

valid_accords = (
    F.col("main_accords").isNotNull()
    & (accords_clean != "")
    & (~F.lower(accords_clean).isin(
        "null", "none", "nan", "n/a", "[]"
    ))
)

accord_array = F.from_json(
    F.regexp_replace(
        F.regexp_replace(F.col("main_accords"), "'", '"'),
        r"\bNone\b",
        "null"
    ),
    "array<string>"
)

# Nasomatto accord data.
nasomatto_accords = (
    nasomatto_rated
    .filter(valid_accords)
    .withColumn("accords", accord_array)
    .withColumn(
        "accords",
        F.transform(
            "accords",
            lambda x: F.lower(F.trim(x))
        )
    )
    .withColumn("dominant_accord", F.get("accords", 0))
    .withColumn("second_accord", F.get("accords", 1))
    .withColumn("third_accord", F.get("accords", 2))
    .filter(
        F.col("gender_key").isin("men", "unisex", "women")
        & F.col("dominant_accord").isNotNull()
    )
)

# Full industry reference population.
industry_accords = (
    df_fragrantica
    .filter(
        valid_accords
        & F.col("rating_value").isNotNull()
        & F.col("rating_count").isNotNull()
        & (F.col("rating_count") >= 0)
        & F.lower(F.trim(F.col("gender"))).isin(
            "men", "unisex", "women"
        )
    )
    .withColumn("accords", accord_array)
    .withColumn(
        "accords",
        F.transform(
            "accords",
            lambda x: F.lower(F.trim(x))
        )
    )
    .withColumn(
        "gender_key",
        F.lower(F.trim(F.col("gender")))
    )
    .withColumn("dominant_accord", F.get("accords", 0))
    .withColumn("second_accord", F.get("accords", 1))
    .withColumn("third_accord", F.get("accords", 2))
    .filter(F.col("dominant_accord").isNotNull())
)

In [0]:
# INDUSTRY BENCHMARKS AT THREE ACCORD LEVELS
# For each accord level, calculate gender-matched medians in two ways:
# (1) across all industry fragrances with valid data at that accord level;
# (2) within each exact ordered accord profile.
# Count distinct fragrance IDs in each reference group. The profile N
# indicates how many industry fragrances support each exact-profile median.

def calculate_benchmarks(data, profile_columns):

    valid_data = data.filter(
        F.col("id").isNotNull()
        & F.col("rating_value").isNotNull()
        & F.col("rating_count").isNotNull()
    )

    gender_benchmark = (
        valid_data
        .groupBy("gender_key")
        .agg(
            F.countDistinct("id").alias("industry_gender_n"),
            F.percentile_approx(
                "rating_value", 0.5, 10000
            ).alias("industry_median_rating"),
            F.percentile_approx(
                "rating_count", 0.5, 10000
            ).alias("industry_median_votes")
        )
    )

    profile_benchmark = (
        valid_data
        .groupBy("gender_key", *profile_columns)
        .agg(
            F.countDistinct("id").alias("profile_reference_n"),
            F.percentile_approx(
                "rating_value", 0.5, 10000
            ).alias("profile_median_rating"),
            F.percentile_approx(
                "rating_count", 0.5, 10000
            ).alias("profile_median_votes")
        )
    )

    return gender_benchmark, profile_benchmark


# Dominant accord.
dominant_gender_benchmark, dominant_profile_benchmark = (
    calculate_benchmarks(
        industry_accords.filter(
            F.col("dominant_accord").isNotNull()
        ),
        ["dominant_accord"]
    )
)

# Ordered dyad.
dyad_gender_benchmark, dyad_profile_benchmark = (
    calculate_benchmarks(
        industry_accords.filter(
            F.col("dominant_accord").isNotNull()
            & F.col("second_accord").isNotNull()
        ),
        ["dominant_accord", "second_accord"]
    )
)

# Ordered triad.
triad_gender_benchmark, triad_profile_benchmark = (
    calculate_benchmarks(
        industry_accords.filter(
            F.col("dominant_accord").isNotNull()
            & F.col("second_accord").isNotNull()
            & F.col("third_accord").isNotNull()
        ),
        ["dominant_accord", "second_accord", "third_accord"]
    )
)

In [0]:
# INDIVIDUAL FRAGRANCE BENCHMARKING - industry level analysis
# Join each Nasomatto fragrance to its gender-wide industry benchmark and
# its exact accord-profile benchmark. Run this for dominant accords, ordered
# dyads and ordered triads, then combine the results into one long-format
# table. Each fragrance can contribute up to three rows, one per level.

def build_fragrance_benchmark(
    brand_data,
    gender_benchmark,
    profile_benchmark,
    profile_columns,
    accord_level
):

    b = brand_data.alias("b")
    g = gender_benchmark.alias("g")
    p = profile_benchmark.alias("p")

    join_columns = ["gender_key"] + profile_columns

    result = (
        b
        .join(g, "gender_key", "inner")
        .join(p, join_columns, "inner")
        .select(
            F.col("b.id").alias("id"),
            F.col("b.fragrance_name_clean").alias("fragrance"),
            F.col("b.gender_key").alias("gender"),
            F.lit(accord_level).alias("accord_level"),

            F.concat_ws(
                " + ",
                *[F.col(f"b.{column}") for column in profile_columns]
            ).alias("accord_profile"),

            F.col("b.rating_value").alias("individual_rating"),
            F.col("b.rating_count").alias("individual_votes"),

            F.col("g.industry_gender_n"),
            F.col("g.industry_median_rating"),
            F.col("g.industry_median_votes"),

            F.col("p.profile_reference_n"),
            F.col("p.profile_median_rating"),
            F.col("p.profile_median_votes")
        )
    )

    return result


dominant_results = build_fragrance_benchmark(
    nasomatto_accords.filter(
        F.col("dominant_accord").isNotNull()
    ),
    dominant_gender_benchmark,
    dominant_profile_benchmark,
    ["dominant_accord"],
    "Dominant accord"
)

dyad_results = build_fragrance_benchmark(
    nasomatto_accords.filter(
        F.col("dominant_accord").isNotNull()
        & F.col("second_accord").isNotNull()
    ),
    dyad_gender_benchmark,
    dyad_profile_benchmark,
    ["dominant_accord", "second_accord"],
    "Dyad"
)

triad_results = build_fragrance_benchmark(
    nasomatto_accords.filter(
        F.col("dominant_accord").isNotNull()
        & F.col("second_accord").isNotNull()
        & F.col("third_accord").isNotNull()
    ),
    triad_gender_benchmark,
    triad_profile_benchmark,
    ["dominant_accord", "second_accord", "third_accord"],
    "Triad"
)

fragrance_benchmark_results = (
    dominant_results
    .unionByName(dyad_results)
    .unionByName(triad_results)
)

display(
    fragrance_benchmark_results
    .orderBy("gender", "fragrance", "accord_level")
)

In [0]:
# NASOMATTO INTERNAL BENCHMARKS - brand level analysis
# Calculate portfolio-wide medians across all rated Nasomatto fragrances,
# regardless of gender, and separate medians by gender. These are the
# reference values for the brand's own portfolio segmentation.

# Overall Nasomatto benchmark.
nasomatto_overall_benchmark = (
    nasomatto_rated
    .agg(
        F.countDistinct("id").alias("brand_reference_n"),
        F.percentile_approx(
            "rating_value", 0.5, 10000
        ).alias("brand_median_rating"),
        F.percentile_approx(
            "rating_count", 0.5, 10000
        ).alias("brand_median_votes")
    )
)

# Gender-specific Nasomatto benchmarks.
nasomatto_gender_benchmark = (
    nasomatto_rated
    .filter(
        F.col("gender_key").isin("men", "unisex", "women")
    )
    .groupBy("gender_key")
    .agg(
        F.countDistinct("id").alias("brand_gender_n"),
        F.percentile_approx(
            "rating_value", 0.5, 10000
        ).alias("brand_gender_median_rating"),
        F.percentile_approx(
            "rating_count", 0.5, 10000
        ).alias("brand_gender_median_votes")
    )
)

display(nasomatto_overall_benchmark)
display(nasomatto_gender_benchmark)

In [0]:
# SHARED FOUR-SEGMENT CLASSIFICATION
# Compare a fragrance's rating and vote count with the two medians supplied
# for the relevant comparison group. Rating measures relative appreciation;
# vote count measures relative community attention. The same rule is reused
# for brand, gender and exact accord-profile segmentation.

def assign_segment(rating, votes, median_rating, median_votes):

    return (
        F.when(
            (F.col(rating) >= F.col(median_rating))
            & (F.col(votes) >= F.col(median_votes)),
            "Standout Profiles"
        )
        .when(
            (F.col(rating) >= F.col(median_rating))
            & (F.col(votes) < F.col(median_votes)),
            "Hidden Favourites"
        )
        .when(
            (F.col(rating) < F.col(median_rating))
            & (F.col(votes) >= F.col(median_votes)),
            "Widely Explored"
        )
        .otherwise("Quiet Profiles")
    )


# Overall portfolio segmentation.
nasomatto_overall_segments = (
    nasomatto_rated
    .crossJoin(nasomatto_overall_benchmark)
    .withColumn(
        "performance_segment",
        assign_segment(
            "rating_value",
            "rating_count",
            "brand_median_rating",
            "brand_median_votes"
        )
    )
)

display(
    nasomatto_overall_segments
    .select(
        "fragrance_name_clean",
        "gender",
        "rating_value",
        "rating_count",
        "performance_segment"
    )
    .orderBy("performance_segment", F.desc("rating_count"))
)

In [0]:
# GENDER-SPECIFIC BRAND SEGMENTATION
# Classify each Nasomatto fragrance against the rating and vote medians
# of its own gender within the Nasomatto portfolio. This complements the
# overall brand segmentation; retain the gender sample size for context.

nasomatto_gender_segments = (
    nasomatto_rated
    .filter(
        F.col("gender_key").isin("men", "unisex", "women")
    )
    .join(
        nasomatto_gender_benchmark,
        "gender_key",
        "inner"
    )
    .withColumn(
        "gender_performance_segment",
        assign_segment(
            "rating_value",
            "rating_count",
            "brand_gender_median_rating",
            "brand_gender_median_votes"
        )
    )
)

display(
    nasomatto_gender_segments
    .select(
        "fragrance_name_clean",
        "gender",
        "rating_value",
        "rating_count",
        "brand_gender_n",
        "gender_performance_segment"
    )
    .orderBy("gender", "gender_performance_segment")
)

In [0]:
# EXACT ACCORD-PROFILE SEGMENTATION
# Classify each Nasomatto fragrance at each available accord level against
# industry fragrances with the exact same ordered accord profile and gender.
# Unlike the gender-wide industry segmentation, this comparison uses only
# direct accord-profile peers. Keep profile_reference_n visible because
# the strength of the comparison depends on the size of that reference group.
# No minimum-N filter is applied; small groups remain visible for review.

nasomatto_exact_profile_segments = (
    fragrance_benchmark_results
    .withColumn(
        "profile_performance_segment",
        assign_segment(
            "individual_rating",
            "individual_votes",
            "profile_median_rating",
            "profile_median_votes"
        )
    )
)

display(
    nasomatto_exact_profile_segments
    .select(
        "fragrance",
        "gender",
        "accord_level",
        "accord_profile",
        "individual_rating",
        "individual_votes",
        "profile_reference_n",
        "profile_median_rating",
        "profile_median_votes",
        "profile_performance_segment"
    )
    .orderBy("gender", "fragrance", "accord_level")
)

# Summarize the number of distinct Nasomatto fragrances in each segment,
# separately by gender and accord level.
nasomatto_exact_profile_segment_summary = (
    nasomatto_exact_profile_segments
    .groupBy("gender", "accord_level", "profile_performance_segment")
    .agg(
        F.countDistinct("id").alias("fragrance_count")
    )
    .orderBy("gender", "accord_level", "profile_performance_segment")
)

display(nasomatto_exact_profile_segment_summary)


In [0]:
# SEGMENT DISTRIBUTION SUMMARIES
# Count distinct fragrances in each segment for the overall Nasomatto
# portfolio and separately by gender to show the distribution of results.

# Overall portfolio.
nasomatto_overall_segment_summary = (
    nasomatto_overall_segments
    .groupBy("performance_segment")
    .agg(
        F.countDistinct("id").alias("fragrance_count")
    )
    .orderBy("performance_segment")
)

# Gender-specific portfolio.
nasomatto_gender_segment_summary = (
    nasomatto_gender_segments
    .groupBy("gender_key", "gender_performance_segment")
    .agg(
        F.countDistinct("id").alias("fragrance_count")
    )
    .orderBy("gender_key", "gender_performance_segment")
)

display(nasomatto_overall_segment_summary)
display(nasomatto_gender_segment_summary)